# 04 Deep Feature Correlation Analysis

This notebook expands the EDA beyond the initial merge by studying correlations across:

- research-motivated audio delivery features,
- all numeric audio features available in the comprehensive table,
- transcript-derived text features,
- post-earnings stock percent returns.

The goal is exploratory data science, not causal inference. We look for signal candidates, redundancy among features, and interpretable patterns worth modeling later.

## Research-Motivated Audio Feature Groups

Prior speech and paralinguistic analysis often treats vocal delivery as a proxy for arousal, stress, confidence, and clarity. In this dataset we use transparent acoustic features rather than black-box speaker emotion labels:

- **Prosody / pitch:** `pitch_mean`, `pitch_std`. Pitch variation can reflect arousal, emphasis, uncertainty, or speaker dynamics.
- **Energy / loudness:** `energy_mean`, `energy_std`. Energy level and variability can reflect intensity, assertiveness, or delivery consistency.
- **Voice activity:** `voiced_ratio`. A rough proxy for continuous voiced speech vs silence/noise.
- **Zero-crossing and spectral features:** `zcr_*`, `spectral_centroid_*`, `spectral_bandwidth_*`. These capture brightness/noisiness and frequency distribution.
- **MFCC features:** `mfcc_*_mean`, `mfcc_*_std`. Compact timbral descriptors commonly used in speech/audio modeling.
- **Composite indices:** `audio_stress_index`, `audio_confidence_index`, `audio_instability_index`, `vocal_clarity_proxy`. These are engineered from standardized acoustic features for exploratory interpretation.

We supplement these with text features from transcripts and stock outcome columns measured as **percent returns**.

In [ ]:
from pathlib import Path
import math

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 160)
pd.set_option("display.max_colwidth", 120)

try:
    from scipy import stats
    SCIPY_AVAILABLE = True
except Exception:
    SCIPY_AVAILABLE = False

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
REPORTS_DIR = PROJECT_ROOT / "reports"

ANALYSIS_PATH = PROCESSED_DIR / "earnings_analysis_table.csv"
AUDIO_DEDUPED_PATH = PROCESSED_DIR / "audio_call_feature_table_deduped.csv"
TEXT_FEATURES_PATH = PROCESSED_DIR / "earnings_text_features.csv"

OUTCOME_CORR_OUTPUT = PROCESSED_DIR / "deep_feature_return_correlations.csv"
FULL_CORR_OUTPUT = PROCESSED_DIR / "deep_numeric_feature_correlation_matrix.csv"
AUDIO_TEXT_CORR_OUTPUT = PROCESSED_DIR / "deep_audio_text_correlation_pairs.csv"
FEATURE_GROUP_SUMMARY_OUTPUT = PROCESSED_DIR / "deep_feature_group_summary.csv"

analysis = pd.read_csv(ANALYSIS_PATH)
audio = pd.read_csv(AUDIO_DEDUPED_PATH)
text_features = pd.read_csv(TEXT_FEATURES_PATH)

analysis.shape, audio.shape, text_features.shape, SCIPY_AVAILABLE

In [ ]:
OUTCOME_COLS = ["return_1d_pct", "return_5d_pct", "return_10d_pct"]
ID_COLS = {
    "base_call_id", "duplicate_suffix", "has_duplicate_suffix", "call_id", "ticker", "call_date", "merge_key",
    "text_call_key", "company", "quarter", "year", "period", "source_url", "scraped_date", "clean_transcript",
    "stock_status", "pre_call_trading_date", "pre_call_close", "has_text_match", "has_stock_match",
}

AUDIO_FEATURE_GROUPS = {
    "prosody_pitch": ["pitch_mean", "pitch_std"],
    "energy_loudness": ["energy_mean", "energy_std"],
    "voice_activity": ["voiced_ratio"],
    "zero_crossing": ["zcr_mean", "zcr_std"],
    "spectral_shape": [
        "spectral_centroid_mean", "spectral_centroid_std",
        "spectral_bandwidth_mean", "spectral_bandwidth_std",
    ],
    "audio_composites": [
        "audio_stress_index", "audio_confidence_index", "audio_instability_index", "vocal_clarity_proxy",
    ],
}

mfcc_mean_cols = [col for col in analysis.columns if col.startswith("mfcc_") and col.endswith("_mean")]
mfcc_std_cols = [col for col in analysis.columns if col.startswith("mfcc_") and col.endswith("_std")]
AUDIO_FEATURE_GROUPS["mfcc_means"] = sorted(mfcc_mean_cols, key=lambda x: int(x.split("_")[1]))
AUDIO_FEATURE_GROUPS["mfcc_variability"] = sorted(mfcc_std_cols, key=lambda x: int(x.split("_")[1]))

TEXT_FEATURE_COLS = [
    "transcript_chars", "word_count", "sentence_count", "avg_sentence_length", "avg_word_length",
    "unique_word_ratio", "positive_term_count", "negative_term_count", "finance_term_count",
    "positive_term_rate", "negative_term_rate", "finance_term_rate", "simple_sentiment_balance",
]
TEXT_FEATURE_COLS = [col for col in TEXT_FEATURE_COLS if col in analysis.columns]

AUDIO_FEATURE_COLS = []
for cols in AUDIO_FEATURE_GROUPS.values():
    AUDIO_FEATURE_COLS.extend([col for col in cols if col in analysis.columns])
AUDIO_FEATURE_COLS = list(dict.fromkeys(AUDIO_FEATURE_COLS))

numeric_cols = analysis.select_dtypes(include="number").columns.tolist()
analysis_feature_cols = [
    col for col in numeric_cols
    if col not in OUTCOME_COLS and col not in ID_COLS and not col.startswith("price_date_")
]

feature_inventory = pd.DataFrame(
    [
        {"group": group, "feature_count": len([col for col in cols if col in analysis.columns]), "features": ", ".join([col for col in cols if col in analysis.columns])}
        for group, cols in AUDIO_FEATURE_GROUPS.items()
    ] + [
        {"group": "text_features", "feature_count": len(TEXT_FEATURE_COLS), "features": ", ".join(TEXT_FEATURE_COLS)},
        {"group": "all_numeric_candidate_features", "feature_count": len(analysis_feature_cols), "features": ", ".join(analysis_feature_cols)},
    ]
)
feature_inventory

In [ ]:
def pearson_spearman(x: pd.Series, y: pd.Series) -> dict[str, float]:
    pair = pd.concat([pd.to_numeric(x, errors="coerce"), pd.to_numeric(y, errors="coerce")], axis=1).dropna()
    n = len(pair)
    if n < 3 or pair.iloc[:, 0].nunique() < 2 or pair.iloc[:, 1].nunique() < 2:
        return {"n": n, "pearson_r": np.nan, "pearson_p": np.nan, "spearman_r": np.nan, "spearman_p": np.nan}

    if SCIPY_AVAILABLE:
        pearson_r, pearson_p = stats.pearsonr(pair.iloc[:, 0], pair.iloc[:, 1])
        spearman_r, spearman_p = stats.spearmanr(pair.iloc[:, 0], pair.iloc[:, 1])
    else:
        pearson_r = pair.iloc[:, 0].corr(pair.iloc[:, 1], method="pearson")
        spearman_r = pair.iloc[:, 0].corr(pair.iloc[:, 1], method="spearman")
        pearson_p = np.nan
        spearman_p = np.nan

    return {
        "n": n,
        "pearson_r": pearson_r,
        "pearson_p": pearson_p,
        "spearman_r": spearman_r,
        "spearman_p": spearman_p,
    }


def bh_adjust(p_values: pd.Series) -> pd.Series:
    p = pd.to_numeric(p_values, errors="coerce")
    out = pd.Series(np.nan, index=p.index, dtype=float)
    valid = p.dropna().sort_values()
    m = len(valid)
    if m == 0:
        return out
    ranked = valid.reset_index()
    ranked["rank"] = np.arange(1, m + 1)
    ranked["q"] = ranked.iloc[:, 1] * m / ranked["rank"]
    ranked["q"] = ranked["q"][::-1].cummin()[::-1].clip(upper=1.0)
    out.loc[ranked["index"]] = ranked["q"].values
    return out


def feature_group(feature: str) -> str:
    for group, cols in AUDIO_FEATURE_GROUPS.items():
        if feature in cols:
            return group
    if feature in TEXT_FEATURE_COLS:
        return "text_features"
    return "other_numeric_features"


def build_outcome_correlations(df: pd.DataFrame, features: list[str], outcomes: list[str]) -> pd.DataFrame:
    rows = []
    for feature in features:
        for outcome in outcomes:
            if feature not in df.columns or outcome not in df.columns:
                continue
            stats_row = pearson_spearman(df[feature], df[outcome])
            rows.append(
                {
                    "feature": feature,
                    "feature_group": feature_group(feature),
                    "outcome": outcome,
                    **stats_row,
                    "abs_pearson_r": abs(stats_row["pearson_r"]) if pd.notna(stats_row["pearson_r"]) else np.nan,
                    "abs_spearman_r": abs(stats_row["spearman_r"]) if pd.notna(stats_row["spearman_r"]) else np.nan,
                }
            )
    corr_df = pd.DataFrame(rows)
    if not corr_df.empty:
        corr_df["pearson_q"] = bh_adjust(corr_df["pearson_p"])
        corr_df["spearman_q"] = bh_adjust(corr_df["spearman_p"])
        corr_df["pearson_fdr_10pct"] = corr_df["pearson_q"].le(0.10)
        corr_df["spearman_fdr_10pct"] = corr_df["spearman_q"].le(0.10)
    return corr_df

len(analysis_feature_cols), len(AUDIO_FEATURE_COLS), len(TEXT_FEATURE_COLS)

In [ ]:
# Full matrix across all numeric candidate features + percent-return outcomes.
matrix_cols = [col for col in analysis_feature_cols + OUTCOME_COLS if col in analysis.columns]
full_corr_matrix = analysis[matrix_cols].corr(method="pearson", numeric_only=True)
full_corr_matrix.to_csv(FULL_CORR_OUTPUT)

outcome_corr = build_outcome_correlations(analysis, analysis_feature_cols, OUTCOME_COLS)
outcome_corr = outcome_corr.sort_values(["outcome", "abs_spearman_r"], ascending=[True, False]).reset_index(drop=True)
outcome_corr.to_csv(OUTCOME_CORR_OUTPUT, index=False)

print(f"Wrote full numeric correlation matrix: {FULL_CORR_OUTPUT}")
print(f"Wrote ranked feature-return correlations: {OUTCOME_CORR_OUTPUT}")
outcome_corr.head(15)

In [ ]:
top_by_outcome = (
    outcome_corr.sort_values(["outcome", "abs_spearman_r"], ascending=[True, False])
    .groupby("outcome", group_keys=False)
    .head(12)
    [[
        "outcome", "feature_group", "feature", "n", "pearson_r", "pearson_p", "pearson_q",
        "spearman_r", "spearman_p", "spearman_q", "abs_spearman_r",
    ]]
)

top_by_outcome

In [ ]:
research_audio_corr = outcome_corr[outcome_corr["feature"].isin(AUDIO_FEATURE_COLS)].copy()
research_audio_corr = research_audio_corr.sort_values(["outcome", "abs_spearman_r"], ascending=[True, False])

research_audio_top = (
    research_audio_corr.groupby("outcome", group_keys=False)
    .head(12)
    [["outcome", "feature_group", "feature", "n", "pearson_r", "spearman_r", "spearman_q", "abs_spearman_r"]]
)
research_audio_top

In [ ]:
group_summary = (
    outcome_corr.groupby(["outcome", "feature_group"], as_index=False)
    .agg(
        features_tested=("feature", "nunique"),
        max_abs_spearman=("abs_spearman_r", "max"),
        median_abs_spearman=("abs_spearman_r", "median"),
        min_spearman_q=("spearman_q", "min"),
        fdr_10pct_hits=("spearman_fdr_10pct", "sum"),
    )
    .sort_values(["outcome", "max_abs_spearman"], ascending=[True, False])
)

group_summary.to_csv(FEATURE_GROUP_SUMMARY_OUTPUT, index=False)
print(f"Wrote feature group summary: {FEATURE_GROUP_SUMMARY_OUTPUT}")
group_summary

In [ ]:
audio_text_rows = []
for audio_feature in AUDIO_FEATURE_COLS:
    for text_feature in TEXT_FEATURE_COLS:
        if audio_feature not in analysis.columns or text_feature not in analysis.columns:
            continue
        stats_row = pearson_spearman(analysis[audio_feature], analysis[text_feature])
        audio_text_rows.append(
            {
                "audio_feature": audio_feature,
                "audio_group": feature_group(audio_feature),
                "text_feature": text_feature,
                **stats_row,
                "abs_spearman_r": abs(stats_row["spearman_r"]) if pd.notna(stats_row["spearman_r"]) else np.nan,
            }
        )

audio_text_corr = pd.DataFrame(audio_text_rows)
if not audio_text_corr.empty:
    audio_text_corr["spearman_q"] = bh_adjust(audio_text_corr["spearman_p"])
    audio_text_corr = audio_text_corr.sort_values("abs_spearman_r", ascending=False)

audio_text_corr.to_csv(AUDIO_TEXT_CORR_OUTPUT, index=False)
print(f"Wrote audio-text correlation pairs: {AUDIO_TEXT_CORR_OUTPUT}")
audio_text_corr.head(20)

In [ ]:
# Pivot top Spearman correlations for easier visual inspection.
top_features_for_heatmap = (
    outcome_corr.sort_values("abs_spearman_r", ascending=False)
    .drop_duplicates("feature")
    .head(30)["feature"]
    .tolist()
)
heatmap_data = (
    outcome_corr[outcome_corr["feature"].isin(top_features_for_heatmap)]
    .pivot(index="feature", columns="outcome", values="spearman_r")
    .reindex(top_features_for_heatmap)
)
heatmap_data

In [ ]:
try:
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(8, max(8, 0.28 * len(heatmap_data))))
    im = ax.imshow(heatmap_data.fillna(0).values, aspect="auto", vmin=-0.5, vmax=0.5, cmap="coolwarm")
    ax.set_xticks(range(len(heatmap_data.columns)))
    ax.set_xticklabels(heatmap_data.columns, rotation=30, ha="right")
    ax.set_yticks(range(len(heatmap_data.index)))
    ax.set_yticklabels(heatmap_data.index)
    ax.set_title("Top feature Spearman correlations with stock percent returns")
    ax.set_xlabel("Outcome")
    ax.set_ylabel("Feature")
    fig.colorbar(im, ax=ax, label="Spearman rho")
    plt.tight_layout()
except ImportError:
    print("matplotlib is not installed; skipping heatmap plot.")

In [ ]:
# Feature redundancy: which audio variables largely move together?
audio_corr_matrix = analysis[AUDIO_FEATURE_COLS].corr(method="spearman", numeric_only=True)
upper = audio_corr_matrix.where(np.triu(np.ones(audio_corr_matrix.shape), k=1).astype(bool))
redundant_audio_pairs = (
    upper.stack()
    .rename("spearman_r")
    .reset_index()
    .rename(columns={"level_0": "feature_a", "level_1": "feature_b"})
)
redundant_audio_pairs["abs_spearman_r"] = redundant_audio_pairs["spearman_r"].abs()
redundant_audio_pairs = redundant_audio_pairs.sort_values("abs_spearman_r", ascending=False)

redundant_audio_pairs.head(20)

In [ ]:
# Compact interpretation helpers for the report.
reportable_top = (
    outcome_corr.sort_values("abs_spearman_r", ascending=False)
    [["outcome", "feature_group", "feature", "n", "spearman_r", "spearman_p", "spearman_q", "pearson_r"]]
    .head(25)
)

reportable_audio = (
    research_audio_corr.sort_values("abs_spearman_r", ascending=False)
    [["outcome", "feature_group", "feature", "n", "spearman_r", "spearman_q", "pearson_r"]]
    .head(25)
)

print("Top all-feature correlations with return outcomes")
display(reportable_top)
print("Top research-motivated audio correlations with return outcomes")
display(reportable_audio)
print("Strongest audio-text relationships")
display(audio_text_corr.head(15))

## Reading These Results

Use the ranked tables as signal screens, not proof. Important points:

- `n` differs for text features because only matched transcript rows can contribute to text correlations.
- `spearman_r` is often more useful than Pearson here because earnings reactions are event-driven and heavy-tailed.
- `spearman_q` applies a Benjamini-Hochberg multiple-testing correction. With this sample size, many interesting EDA effects may not survive correction.
- High audio-audio correlations suggest redundant features. Modeling should reduce dimensionality, regularize, or choose representative features by family.
- Percent return targets are the primary stock outcomes; absolute price movement is intentionally not used.